In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.mps.is_available():
    device = torch.device("mps")
print(f"Using device: {device}")

Using device: mps


In [ ]:
import json
with open("data/traces.jsonl", "r") as f:
    traces = [json.loads(line) for line in f.readlines()]
import pandas as pd

df = pd.DataFrame(traces)

In [4]:
from src.constants import TOP_RFHS_BY_LAYER_HEAD

# Transform TOP_RFHS_BY_LAYER_HEAD to map layer -> [rfh_indices] for each model
layer_to_top_rfhs = {}
for k, v in TOP_RFHS_BY_LAYER_HEAD.items():
    d_tmp = {}
    for cord in v:
        if cord[0] not in d_tmp:
            d_tmp[cord[0]] = []
        d_tmp[cord[0]].append(cord[1])
    layer_to_top_rfhs[k] = d_tmp
layer_to_top_rfhs["qwen-1p5B"]

{16: [2, 11, 0], 1: [5], 19: [1, 5], 12: [1], 23: [2], 14: [3], 20: [9]}

In [5]:
from src.patcher import ActivationPatcher, clear_memory
from src.constants import MODELS_LITERAL

model_alias: MODELS_LITERAL = "qwen-1p5B"

# instantiate patcher
patcher = ActivationPatcher(model_alias, device)

/Users/rishidinesh/Projects/causal-mediation-analysis-rfh/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B into HookedTransformer


In [6]:
# filter dataframe and layer_rfh map for the specific model
df = df[df["model_name"] == model_alias].reset_index(drop=True)
layer_to_top_rfhs = layer_to_top_rfhs[model_alias]
layer_to_top_rfhs

{16: [2, 11, 0], 1: [5], 19: [1, 5], 12: [1], 23: [2], 14: [3], 20: [9]}

In [7]:
import os
from pathlib import Path

def run_patching(df: pd.DataFrame, heads_by_layer: dict, savepath: str, save_frequency: int = 1):
    results = []
    os.makedirs(Path(savepath).parent, exist_ok=True)
    for i, row in df.iterrows():
        row = row.to_dict()
        try:
            res = patcher.run(
                response_withR = row["response_withR"],
                response_withoutR = row["response_withoutR"],
                heads_by_layer = heads_by_layer
            )
        except RuntimeError as e:
            print(f"RuntimeError at index {i}, skipping. Error: {e}")
            clear_memory()
            continue
        print(f"{i} | ID: {row['unique_id']} | withR_loss: {res['withR_loss']:.4f} | withoutR_loss: {res['withoutR_loss']:.4f} | patched_withR_loss: {res['patched_withR_loss']:.4f} | patched_withoutR_loss: {res['patched_withoutR_loss']:.4f}")
        res.update({
            "unique_id": row["unique_id"],
            "model": row["model_name"]
        })
        results.append(res)
        if (i + 1) % save_frequency == 0:
            # print(f"Processed {i + 1} examples, saving intermediate results to {savepath}")
            with open(savepath, "a") as f:
                for r in results:
                    f.write(json.dumps(r) + "\n")
            results = []
    # save any remaining results
    if results:
        print(f"Saving final results to {savepath}")
        with open(savepath, "a") as f:
            for r in results:
                f.write(json.dumps(r) + "\n")
    results = []

### Patch All RFHs

In [ ]:
run_patching(
    df = df,
    heads_by_layer = layer_to_top_rfhs,
    savepath = f"data/output/activation_patching/{model_alias}/all_rfh_heads.jsonl"
)

### Patch Layer-wise

In [ ]:
for layer in layer_to_top_rfhs.keys():
    heads_by_layer = {layer: layer_to_top_rfhs[layer]}
    run_patching(
        df = df,
        heads_by_layer = heads_by_layer,
        savepath = f"data/output/activation_patching/{model_alias}/layer_{layer}_rfh_heads.jsonl"
    )

### Patch Individual RFH

In [ ]:
for layer in layer_to_top_rfhs.keys():
    for head in layer_to_top_rfhs[layer]:
        heads_by_layer = {layer: head}
        run_patching(
            df = df,
            heads_by_layer = heads_by_layer,
            savepath = f"data/output/activation_patching/{model_alias}/layer_{layer}_rfh_head_{head}.jsonl"
        )

### Patch Top-K RFH

In [ ]:
# for qwen-1.5B, we identify the topK RFH from the previous run
qwen_1p5_top_rfh = [(23, 2), (16, 2), (19, 1), (14, 3), (20, 9), (19, 5)]

for k in range(1, 6):
    top_k = qwen_1p5_top_rfh[:k]
    heads_by_layer: dict[int, list[int]] = {}
    for l, h in top_k:
        if l not in heads_by_layer:
            heads_by_layer[l] = []
        heads_by_layer[l].append(h)
    run_patching(
        df = df,
        heads_by_layer = heads_by_layer,
        savepath = f"data/output/activation_patching/{model_alias}/top_{k}_rfh.jsonl"
    )

{23: [2]}
{23: [2], 16: [2]}
{23: [2], 16: [2], 19: [1]}
{23: [2], 16: [2], 19: [1], 14: [3]}
{23: [2], 16: [2], 19: [1], 14: [3], 20: [9]}
